# 3D-RCNN, in miniature: instance 3D shape + pose by **render-and-compare**

A faithful, self-contained miniature of *3D-RCNN* (Kundu, Li, Rehg, CVPR 2018):

1. a low-dimensional **PCA shape basis** over a collection of 3D objects (the shape *modes*);
2. a CNN that maps an **image → (shape code, pose)**;
3. a **render-and-compare loss** — render the predicted shape at the predicted pose to a
   silhouette and compare to the 2D mask, so **2D supervision trains 3D**.

Modernised vs the paper: a **differentiable** voxel renderer (true backprop) instead of the
paper's finite-difference gradients; single object, no detection backbone. Runs on CPU or GPU.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); print('device:', dev)

## 1. Procedural 3D objects (surface point clouds), normalised to a unit sphere

In [ ]:
def _grid(nu, nv):
    u=(np.arange(nu)+0.5)/nu; v=(np.arange(nv)+0.5)/nv
    uu,vv=np.meshgrid(u,v,indexing='ij'); return uu.ravel(), vv.ravel()
def sphere(n=48):
    u,v=_grid(n,n); th=2*np.pi*u; ph=np.pi*v
    return np.stack([np.sin(ph)*np.cos(th), np.cos(ph), np.sin(ph)*np.sin(th)],1)
def ellipsoid(a,b,c,n=48): return sphere(n)*np.array([a,b,c])
def box(sx,sy,sz,n=30):
    u,v=_grid(n,n); a,b=u*2-1,v*2-1; P=[]
    for ax in range(3):
        for sg in (-1,1):
            q=np.zeros((len(a),3)); o=[i for i in range(3) if i!=ax]
            q[:,ax]=sg; q[:,o[0]]=a; q[:,o[1]]=b; P.append(q*np.array([sx,sy,sz]))
    return np.concatenate(P)
def cylinder(r,h,n=48):
    u,v=_grid(n,n); th=2*np.pi*u; y=(v*2-1)*h
    return np.stack([r*np.cos(th), y, r*np.sin(th)],1)
def cone(r,h,n=48):
    u,v=_grid(n,n); th=2*np.pi*u; t=v; rad=r*(1-t)
    return np.stack([rad*np.cos(th), (t*2-1)*h, rad*np.sin(th)],1)
def torus(R,r,n=56):
    u,v=_grid(n,n); a=2*np.pi*u; b=2*np.pi*v; rb=R+r*np.cos(b)
    return np.stack([rb*np.cos(a), r*np.sin(b), rb*np.sin(a)],1)
def _norm(p): return p/(np.linalg.norm(p,axis=1).max()+1e-9)
OBJECTS={'sphere':sphere(),'ellipsoid':ellipsoid(.7,1.3,.7),'box':box(.9,.9,.9),
         'slab':box(1.3,.35,.9),'cylinder':cylinder(.7,1.3),'cone':cone(.95,1.4),
         'torus':torus(.9,.35),'rod':box(.32,1.35,.32)}
OBJECTS={k:_norm(v) for k,v in OBJECTS.items()}
NAMES=list(OBJECTS); print(len(NAMES),'objects:',NAMES)

## 2. Shape basis — PCA over the objects' occupancy grids (the shape *modes*)

In [ ]:
D, NC = 20, 8     # voxel resolution, number of shape modes
def voxelize(points, D):
    idx=np.clip(((points+1)/2*D).astype(int),0,D-1)
    occ=np.zeros((D,D,D),np.float32); np.add.at(occ,(idx[:,0],idx[:,1],idx[:,2]),1.0)
    return (occ>0).astype(np.float32)
occ=np.stack([voxelize(OBJECTS[k],D).ravel() for k in NAMES])
mean=occ.mean(0); U,S,Vt=np.linalg.svd(occ-mean,full_matrices=False)
basis=Vt[:NC]; codes=(occ-mean)@basis.T
mean_t=torch.tensor(mean,device=dev); basis_t=torch.tensor(basis,device=dev)
codes_t=torch.tensor(codes,dtype=torch.float32,device=dev)
rec=np.clip(mean+codes@basis,0,1)
print('shape-basis reconstruction MSE:', float(((rec-occ)**2).mean()))

## 3. Differentiable renderer: (shape code, pose) → silhouette

In [ ]:
def rotate3d(z, R):
    theta=torch.zeros(z.size(0),3,4,device=z.device,dtype=z.dtype); theta[:,:3,:3]=R
    grid=F.affine_grid(theta,z.size(),align_corners=False)
    return F.grid_sample(z,grid,align_corners=False,padding_mode='zeros')
def quat_to_R(q):
    q=F.normalize(q,dim=1); w,x,y,z=q[:,0],q[:,1],q[:,2],q[:,3]
    R=torch.stack([1-2*(y*y+z*z),2*(x*y-z*w),2*(x*z+y*w),
                   2*(x*y+z*w),1-2*(x*x+z*z),2*(y*z-x*w),
                   2*(x*z-y*w),2*(y*z+x*w),1-2*(x*x+y*y)],1)
    return R.view(-1,3,3)
H=48
def render_silhouette(code, R):
    occ=(mean_t+code@basis_t).clamp(0,1).view(-1,1,D,D,D)
    occ=rotate3d(occ,R).clamp(0,1)
    sil=1-torch.prod(1-occ,dim=2)                 # soft alpha-composite along depth
    return F.interpolate(sil,size=(H,H),mode='bilinear',align_corners=False)[:,0]

## 4. The network: image → (shape code, pose quaternion)

In [ ]:
def _down(ci,co): return nn.Sequential(nn.Conv2d(ci,co,3,stride=2,padding=1),
                                          nn.BatchNorm2d(co), nn.ReLU())
class R3DCNN(nn.Module):
    def __init__(self,nc):
        super().__init__()
        self.conv=nn.Sequential(_down(1,16),_down(16,32),_down(32,64))
        self.head=nn.Sequential(nn.Linear(64,128),nn.ReLU(),nn.Linear(128,nc+4))
        self.nc=nc
    def forward(self,x):
        h=F.adaptive_avg_pool2d(self.conv(x),1).flatten(1); o=self.head(h)
        return o[:,:self.nc], quat_to_R(o[:,self.nc:])
net=R3DCNN(NC).to(dev)

## 5. Synthetic scenes + render-and-compare training\nInput = a shaded, textured render; target = the 2D silhouette. The loss renders the network's predicted shape+pose and compares to the mask (+ light shape/pose aux).

In [ ]:
def rand_quats(n,rng):
    return F.normalize(torch.tensor(rng.normal(size=(n,4)),dtype=torch.float32,device=dev),dim=1)
@torch.no_grad()
def gen(n, seed):
    rng=np.random.default_rng(seed); o=rng.integers(0,len(NAMES),n)
    gt_code=codes_t[o]; R=quat_to_R(rand_quats(n,rng))
    occ=(mean_t+gt_code@basis_t).clamp(0,1).view(n,1,D,D,D); occ_r=rotate3d(occ,R)
    w=torch.linspace(1.0,0.35,D,device=dev).view(1,1,D,1,1)
    shaded=F.interpolate((occ_r*w).amax(2),size=(H,H),mode='bilinear',align_corners=False)[:,0]
    sil=F.interpolate(1-torch.prod(1-occ_r,2),size=(H,H),mode='bilinear',align_corners=False)[:,0]
    mask=(sil>0.3).float()
    tex=0.6+0.4*torch.tensor(rng.random((n,H,H)),dtype=torch.float32,device=dev)
    img=(shaded*tex+torch.tensor(rng.normal(0,0.04,(n,H,H)),dtype=torch.float32,device=dev)).clamp(0,1)
    return img, mask, gt_code, R
Xtr,Mtr,Ctr,Rtr=gen(900,0); Xte,Mte,Cte,Rte=gen(120,1)
opt=torch.optim.Adam(net.parameters(),lr=1.5e-3); bs=32
for ep in range(70):
    perm=torch.randperm(len(Xtr))
    for i in range(0,len(Xtr),bs):
        idx=perm[i:i+bs]; code,R=net(Xtr[idx].unsqueeze(1))
        sil=render_silhouette(code,R).clamp(1e-4,1-1e-4)
        loss=F.binary_cross_entropy(sil,Mtr[idx])+0.1*F.mse_loss(code,Ctr[idx])+0.1*F.mse_loss(R,Rtr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    if (ep+1)%20==0: print(f'epoch {ep+1}/70  loss={loss.item():.4f}')

## 6. Results — 2D render-and-compare recovered 3D shape + pose

In [ ]:
net.eval()
with torch.no_grad():
    code,R=net(Xte.unsqueeze(1)); sil=render_silhouette(code,R)
    inter=((sil>0.5)&(Mte>0.5)).sum((-1,-2)).float(); union=((sil>0.5)|(Mte>0.5)).sum((-1,-2)).float()
    print('render-and-compare silhouette IoU (test):', float((inter/(union+1e-6)).mean()))
    nv=quat_to_R(rand_quats(6,np.random.default_rng(9))); novel=render_silhouette(code[:6],nv)
rows=[(Xte,'input'),(Mte,'GT mask'),(sil,'pred silhouette (R&C)'),(novel,'pred shape, NOVEL pose')]
fig,ax=plt.subplots(4,6,figsize=(9,6))
for r,(A,lab) in enumerate(rows):
    for c in range(6): ax[r,c].imshow(A[c].cpu().numpy(),cmap='gray',vmin=0,vmax=1); ax[r,c].axis('off')
    ax[r,0].set_title(lab,fontsize=8,loc='left')
plt.suptitle('3D-RCNN miniature: 2D render-and-compare recovers 3D shape+pose'); plt.tight_layout(); plt.show()

**What to read.** Row 3 (the network's render of its predicted shape+pose) matches row 2
(the GT mask) — trained *only* by comparing renders to 2D masks. Row 4 renders the *same*
predicted shape from a **new** pose, showing a 3D model was recovered from one view.

**Honest caveats.** The silhouette underdetermines the full shape code (2D supervision is lossy),
so the *code* is only roughly right even when the silhouette matches — the single-view limit.
Real 3D-RCNN adds a detection backbone, real datasets (Pascal3D+/KITTI), CAD shape bases, and
depth supervision; this is the core idea in miniature.